In [10]:
import pandas as pd, numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib

# load the clean data
df = pd.read_csv("../data/processed/clean.csv",
                 parse_dates=["created_at", "actual_delivery_time"])

df["available_dashers"] = df["total_onshift_dashers"] - df["total_busy_dashers"]
df["busy_ratio"] = df["total_busy_dashers"] / (df["total_onshift_dashers"] + 1)
df["orders_per_dasher"] = df["total_outstanding_orders"] / (df["total_onshift_dashers"] + 1)
df["avg_item_price"] = df["subtotal"] / df["total_items"].clip(lower=1)
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

# sort by time and split: first 80% train, last 20% test
df = df.sort_values("created_at", kind="stable").reset_index(drop=True)
split = int(len(df) * 0.8)

# inputs (X) and answer (y)
X = df.drop(columns=["created_at", "actual_delivery_time", "duration_min", "store_id"])
y = df["duration_min"]
X = pd.get_dummies(X, columns=["market_id", "order_protocol", "store_primary_category"], dtype=int)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(X_train.shape, X_test.shape)

(157826, 108) (39457, 108)


In [11]:
# delayed = slower than 75% of training orders
threshold = y_train.quantile(0.75)
y_train_d = (y_train > threshold).astype(int)
y_test_d = (y_test > threshold).astype(int)

print("threshold (min):", round(threshold, 1))
print("train delayed share:", round(y_train_d.mean(), 3))
print("test delayed share:", round(y_test_d.mean(), 3))

threshold (min): 55.6
train delayed share: 0.25
test delayed share: 0.301


In [12]:

clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                         random_state=42, verbose=-1)
clf.fit(X_train, y_train_d)

# chance of being delayed for each test order
prob = clf.predict_proba(X_test)[:, 1]

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

for t in [0.2, 0.3, 0.35, 0.4, 0.5]:
    p = (prob >= t).astype(int)
    print(t, "| precision:", round(precision_score(y_test_d, p), 3),
          "| recall:", round(recall_score(y_test_d, p), 3),
          "| F1:", round(f1_score(y_test_d, p), 3))

print("PR-AUC:", round(average_precision_score(y_test_d, prob), 3))
print("PR-AUC if guessing:", round(y_test_d.mean(), 3))

0.2 | precision: 0.48 | recall: 0.811 | F1: 0.603
0.3 | precision: 0.568 | recall: 0.66 | F1: 0.611
0.35 | precision: 0.606 | recall: 0.581 | F1: 0.593
0.4 | precision: 0.641 | recall: 0.514 | F1: 0.571
0.5 | precision: 0.704 | recall: 0.374 | F1: 0.488
PR-AUC: 0.638
PR-AUC if guessing: 0.301


In [14]:
from sklearn.metrics import confusion_matrix

# alert if chance of delay is 30% or more
pred = (prob >= 0.3).astype(int)

print("precision:", round(precision_score(y_test_d, pred), 3))
print("recall:", round(recall_score(y_test_d, pred), 3))
print("F1:", round(f1_score(y_test_d, pred), 3))
print(confusion_matrix(y_test_d, pred))

precision: 0.568
recall: 0.66
F1: 0.611
[[21610  5957]
 [ 4043  7847]]


In [15]:
# for each probability bucket: how often were orders really delayed?
bins = pd.cut(prob, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
check = pd.DataFrame({"bucket": bins, "really_delayed": y_test_d.values})
check.groupby("bucket", observed=True)["really_delayed"].agg(["mean", "count"])

,mean,count
bucket,,
"(0.0, 0.2]",0.115908,19343
"(0.2, 0.4]",0.334027,10577
"(0.4, 0.6]",0.561886,5704
"(0.6, 0.8]",0.730457,3083
"(0.8, 1.0]",0.877333,750


In [8]:
qmodels = {}
q_pred = {}
for a in [0.1, 0.5, 0.9]:
    m = lgb.LGBMRegressor(objective="quantile", alpha=a, n_estimators=300,
                          learning_rate=0.05, num_leaves=31,
                          random_state=42, verbose=-1)
    m.fit(X_train, y_train)
    qmodels[a] = m
    q_pred[a] = m.predict(X_test)

# sort each row so low <= mid <= high
s = np.sort(np.column_stack([q_pred[0.1], q_pred[0.5], q_pred[0.9]]), axis=1)
low, mid, high = s[:, 0], s[:, 1], s[:, 2]

# how often is the real time inside the range?
inside = ((y_test.values >= low) & (y_test.values <= high)).mean()
print("real time inside range:", round(inside * 100, 1), "%  (ideal is about 80%)")
print("average range width:", round((high - low).mean(), 1), "min")
print("median model MAE:", round(abs(y_test.values - mid).mean(), 2))

real time inside range: 78.7 %  (ideal is about 80%)
average range width: 32.4 min
median model MAE: 10.93


In [16]:
joblib.dump(clf, "../models/delay_clf.pkl")
joblib.dump(qmodels, "../models/eta_quantiles.pkl")
joblib.dump({"delay_minutes": float(threshold), "alert_probability": 0.3},
            "../models/settings.pkl")
joblib.dump(list(X_train.columns), "../models/feature_columns.pkl")
print("saved")

saved


## Day 3 Findings
1. "Delayed" means slower than 55.6 min (75th percentile of training orders).
2. The delay classifier reached PR-AUC 0.638 vs 0.301 for random guessing.
3. At alert threshold 0.3: precision 0.57, recall 0.66, F1 0.61.
4. Class weighting inflated probabilities, so I trained without it. Probabilities now match real delay rates (scores of 60 to 80% were 73% delayed).
5. The 10th to 90th percentile ETA range contained the real time 78.7% of the time (target 80%), with an average width of 32.4 min.